# Woche 10 – Dienstag: Federated Learning – Simulation des FedAvg‑Algorithmus

## Block 1 (06:00 – 08:45): FedAvg – Theorie

**Lernziel:** Sie verstehen den Algorithmus von McMahan et al. (2017).

Der **Federated Averaging** (FedAvg) Algorithmus:
1. Server initialisiert globale Modellgewichte $w^0$.
2. In jeder Runde $t$: Server wählt eine Teilmenge der Clients aus.
3. Jeder ausgewählte Client trainiert das Modell auf seinen lokalen Daten (mehrere Epochen) und sendet die Gewichtsdifferenz $\Delta w$ zurück.
4. Server aggregiert die Updates (gewichtetes Mittel) und aktualisiert das globale Modell: $w^{t+1} = w^t + \frac{1}{K} \sum \Delta w_k$.

**Aufgabe:** Notieren Sie die Vor‑ und Nachteile von FedAvg gegenüber zentralem Training.

---

## Block 2 (09:30 – 11:40): Python‑Simulation eines einfachen FL mit zwei Clients

**Lernziel:** Sie implementieren eine minimale FL‑Simulation in PyTorch.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# Einfaches lineares Modell
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(2, 1)
    def forward(self, x):
        return self.fc(x)

# Synthetische Daten für zwei Clients
np.random.seed(42)
X_client1 = torch.randn(100, 2)
y_client1 = (X_client1[:,0] + X_client1[:,1] > 0).float().unsqueeze(1)
X_client2 = torch.randn(100, 2)
y_client2 = (X_client2[:,0] + X_client2[:,1] > 0).float().unsqueeze(1)

def train_client(model, X, y, epochs=5, lr=0.01):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
    return model.state_dict()

# Server initialisiert globales Modell
global_model = SimpleModel()

# FL Runden
num_rounds = 5
for round in range(num_rounds):
    # Client 1 trainiert
    client1_model = SimpleModel()
    client1_model.load_state_dict(global_model.state_dict())
    client1_state = train_client(client1_model, X_client1, y_client1)
    # Client 2 trainiert
    client2_model = SimpleModel()
    client2_model.load_state_dict(global_model.state_dict())
    client2_state = train_client(client2_model, X_client2, y_client2)
    # Aggregation (FedAvg)
    new_state = {}
    for key in global_model.state_dict().keys():
        new_state[key] = (client1_state[key] + client2_state[key]) / 2.0
    global_model.load_state_dict(new_state)
    print(f'Runde {round+1} abgeschlossen')

print('Federated Learning abgeschlossen.')


> **Verbesserungshinweis:** In echten Anwendungen werden die Client‑Updates gewichtet nach der Anzahl der lokalen Datenpunkte.

---

## Block 3 (12:40 – 14:10): GOV – Grundrechte‑Folgenabschätzung für FL

**Aufgabe:** Erstellen Sie ein Template für eine Grundrechte‑Folgenabschätzung speziell für ein FL‑System (Art. 27). Berücksichtigen Sie:
- Datenschutz (keine zentrale Datenhaltung, aber Risiko der Modell‑Inversion)
- Sicherheitsrisiken (Poisoning, Free‑riding)
- Fairness (unterschiedliche Datenverteilungen auf Clients)

Speichern Sie das Template als `51_fl_grundrechte_template.md` in Track_C.

---

## Block 4 (14:40 – 16:50): Passiv‑Ersatzformen (Vertiefung)

Buscha Kap. 1.5.3, Übungen zu „sich lassen“ und „sein + zu + Infinitiv“. Schreiben Sie 10 Sätze über FL mit diesen Ersatzformen. Beispiel: „Die Client‑Updates lassen sich ohne Kenntnis der Rohdaten aggregieren.“

Speichern Sie als `52_passiv_ersatz_fl_vertiefung.md` in Track_C.